# All OpenAI Solution

## Installations

In [25]:
!pip install -qq llama-index
!pip install -qq llama-index-llms-azure-openai llama-index-embeddings-azure-openai
!pip install -qq llama-index-readers-file
!pip install -qq llama-index-packs-rag-evaluator

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index-packs-rag-evaluator 0.4.1 requires llama-index-llms-openai<0.6,>=0.5.0, but you have llama-index-llms-openai 0.6.15 which is incompatible.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
llama-index 0.14.13 requires llama-index-llms-openai<0.7,>=0.6.0, but you have llama-index-llms-openai 0.5.6 which is incompatible.
llama-index-cli 0.5.3 requires llama-index-llms-openai<0.7,>=0.6.0, but you have llama-index-llms-openai 0.5.6 which is incompatible.
llama-index-llms-azure-openai 0.4.2 requires llama-index-llms-openai<0.7,>=0.6.0, but you have llama-index-llms-openai 0.5.6 which is incompatible.


## Imports

In [26]:
from pathlib import Path
from llama_index.readers.file import PDFReader
import os

from dotenv import load_dotenv, find_dotenv
from llama_index.llms.azure_openai import AzureOpenAI
from llama_index.core.llms import ChatMessage
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.azure_openai import AzureOpenAIEmbedding
from llama_index.core import VectorStoreIndex
from llama_index.core.llms import ChatMessage

from llama_index.core.ingestion import IngestionPipeline

from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    Response,
)

from llama_index.core.llama_dataset import LabelledRagDataset
from llama_index.packs.rag_evaluator import RagEvaluatorPack
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

from llama_index.core.evaluation import (
    FaithfulnessEvaluator,
    RelevancyEvaluator,
    CorrectnessEvaluator,
    RetrieverEvaluator,
    generate_question_context_pairs,
    EmbeddingQAFinetuneDataset
)

from llama_index.core.llama_dataset.generator import RagDatasetGenerator

## Environment Variables

In [27]:
# load_dotenv('/home/santhosh/Projects/courses/Pinnacle/.env')

os.environ["AZURE_OPENAI_API_KEY"] = "F8kUozKumg8vOqdM6i3uF3MEnHQyAWnh5si5hgocdPdXbakenhTWJQQJ99BLACfhMk5XJ3w3AAAAACOGSVUE"
os.environ["AZURE_OPENAI_ENDPOINT"] = "https://genaifoundry766488650611.openai.azure.com/"
os.environ["OPENAI_API_VERSION"] = "2024-02-01"

## Reading the PDF file

In [28]:
loader = PDFReader()

documents = loader.load_data(file=Path('/content/Final Policy document_LICs New Jeevan Shanti_V05_logo.pdf'))

len(documents)

21

## Creating the Vector Store

In [29]:
embed_model = AzureOpenAIEmbedding(
    model="text-embedding-3-small"
)

In [30]:
embedding = embed_model.get_text_embedding("The cat sat on the mat")

In [31]:
# https://developers.llamaindex.ai/python/framework/module_guides/indexing/vector_store_index/#using-the-ingestion-pipeline-to-create-nodes

# create the pipeline with transformations
pipeline = IngestionPipeline(
    transformations=[
        SentenceSplitter(chunk_size=256, chunk_overlap=20),
        embed_model,
    ]
)

# run the pipeline
nodes = pipeline.run(documents=documents)

len(nodes)

80

In [32]:
vector_index = VectorStoreIndex(
    nodes,
    show_progress=True,
    embed_model=embed_model,
)

Generating embeddings: 0it [00:00, ?it/s]

In [33]:
vector_retriever = vector_index.as_retriever(similarity_top_k=3)

In [34]:
retrieved_nodes = vector_retriever.retrieve("When should we notify the death certificate?")

In [35]:
retrieved_nodes[1].text

'The current provisions of Section 39 are enclosed as \nAnnexure -II for reference. \n \n5) Within 90 days from the date of death, intimation of death along with death certificate must be \nnotified in writing to the office of the Corporation where the policy is serviced for any claims to be \nadmissible. However, delay in intimation of the genuine claim by the claimant may be condoned \nby the Corporation on merit and where delay is proved to be for the reasons beyond his/her \ncontrol. \n \n6) The provisions of Section 45 of Insurance Act, 1938 as amended from time to time shall be \napplicable. The current provisions of the same are enclosed as Annexure-III. \n \n7) Various Sections of the Insurance Act, 1938, applicable to LIC to apply as amended from time to \ntime. \n \n8) The approved version of Policy Document in respect of this Plan is available on our website: \nwww.licindia.in \n \n9) Please visit our website: www.licindia.in to avail LIC’s e-services.'

In [36]:
llm = AzureOpenAI(
    engine="gpt-4o-mini",
    model="gpt-4o-mini",
    temperature=0.0,
)

In [37]:
chat_engine = vector_index.as_chat_engine(chat_mode="context", llm=llm)

In [38]:
response = chat_engine.chat("When should we notify the death certificate?")

In [39]:
print(response)

You must notify the death certificate within 90 days from the date of death. This notification should be made in writing to the office of the Corporation where the policy is serviced for any claim to be admissible.


# Evaluating the model

In [65]:
llm_judge = AzureOpenAI(
    engine="gpt-4o",
    model="gpt-4o",
    temperature=0.0,
)

In [66]:
data_generator = RagDatasetGenerator.from_documents(
    documents,
    llm=llm_judge,
    num_questions_per_chunk=2
)

In [67]:
eval_dataset = data_generator.generate_dataset_from_nodes()

In [68]:
eval_dataset.examples[0].query

'What is the Free Look Period mentioned in the LIC’s New Jeevan Shanti policy, and what steps must a policyholder take if they disagree with the terms and conditions of the policy?'

In [69]:
eval_dataset.examples[0].reference_answer

'The Free Look Period mentioned in the LIC’s New Jeevan Shanti policy is **30 days** from the date of receipt of the electronic or physical mode of the Policy Document, whichever is earlier.\n\nIf a policyholder disagrees with the terms and conditions of the policy, they must:\n\n1. Return the policy within the Free Look Period (30 days).\n2. State the reasons for their objections and disagreement.\n\nUpon receipt of the returned policy, LIC will cancel the policy and refund the premium deposited by the policyholder after deducting charges for stamp duty and any annuity paid (if applicable). For policies under QROPS, the process will also be subject to specific provisions as per the Rules and Regulations of the HMRC.'

In [70]:
eval_questions = [example.query for example in eval_dataset.examples]
eval_answers = [example.reference_answer for example in eval_dataset.examples]

In [71]:
len(eval_answers)

42

In [88]:
# Query Engine
query_engine = vector_index.as_query_engine(llm=llm)
# Create Evaluators
relevancy_evaluator = RelevancyEvaluator(llm=llm)
faithfulness_evaluator = FaithfulnessEvaluator(llm=llm_judge)
correctness_evaluator = CorrectnessEvaluator(llm=llm_judge)

In [89]:
from llama_index.core.evaluation import BatchEvalRunner

runner = BatchEvalRunner(
    {
     "faithfulness": faithfulness_evaluator,
     "relevancy": relevancy_evaluator,
     "correctness": correctness_evaluator
     },
    workers=8,
)

eval_results = await runner.aevaluate_queries(
    query_engine, queries=eval_questions, reference = eval_answers
)

In [90]:
def get_eval_results(key, eval_results):
    results = eval_results[key]
    correct = 0
    for result in results:
        if result.passing:
            correct += 1
    score = correct / len(results)
    print(f"{key} Score: {score}")
    return score

In [92]:
get_eval_results("faithfulness", eval_results)
get_eval_results("relevancy", eval_results)
_ = get_eval_results("correctness", eval_results)

faithfulness Score: 0.9761904761904762
relevancy Score: 0.9761904761904762
correctness Score: 0.8809523809523809
